# IEEE 8500 — chunked dataset with **fixed controller initialization**

Same spacing / scenario rules as the past no-BESS diverse generator.

**Save location (same parent, different folder name):**

| | Path |
|--|------|
| Previous | .../datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40 |
| **This run** | .../datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40_fixedctrlinit |

Parent is auto-picked as the datasets_gnn2 that already contains the previous folder (H: / K: / Colab Drive).

| Knob | Value |
|------|-------|
| Scenarios x samples | 2000 x 40 |
| Chunk size | 50 scenarios |
| Time bins | load/pv/net = 3, include_anchors=True |
| Load/PV scale ranges | [0.3, 1.8] |
| sigma_load / sigma_pv | 0.5 |
| BESS | off |
| Node PE | k=8 from static edges |

**Only change vs the old workflow:** 
ixed_controller_init=True (reset TapNumber=0; CapControl banks OFF; fixed bank (CAPBank3) ON).


In [ ]:
# ============================================================
# CHUNKED RUN: scenarios x samples/scenario — NO BESS
# Same spacing as past diverse 2000×40, PLUS fixed controller init
# (reset: TapNumber=0; CapControl banks OFF; fixed bank ON).
# ============================================================
import os
import math
import time
from pathlib import Path

# ---------------- user controls ----------------
INCLUDE_BESS = False
FIXED_CONTROLLER_INIT = True  # ONLY behavioral change vs past diverse generator

TOTAL_SCENARIOS = 2000
N_SAMPLES_PER_SCENARIO = 40
SCENARIOS_PER_CHUNK = 50
BASE_SEED = 20320230

SIGMA_LOAD = 0.5
SIGMA_PV = 0.5

P_LOAD_MEAN_KW = 13731.9
Q_LOAD_MEAN_KVAR = 2610.15
P_LOAD_SCALE_RANGE = (0.3, 1.8)
Q_LOAD_SCALE_RANGE = (0.3, 1.8)
P_PV_SCALE_RANGE = (0.3, 1.8)

# Only used if INCLUDE_BESS is True
BESS_TOTAL_MVA_MEAN = 0
BESS_TOTAL_MVA_SIGMA = 0.1
BESS_NUM_NODES_MIN = 1
BESS_NUM_NODES_MAX = 3
BESS_Q_FRAC_MAX = 0.44
BESS_CANDIDATE_NODES_150 = []  # fill if you set INCLUDE_BESS = True

VMIN_SAFE_PU = 0.55
VMAX_SAFE_PU = 1.45

NODE_PE_K = 8
NODE_PE_SEED = 42
NODE_PE_ZERO_EIG_TOL = 1e-8

RETURN_NODE_DF = False

# Same parent directory as the previous diverse dataset; different folder name only.
# Previous:  .../datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40
# This run:  .../datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40_fixedctrlinit
_PREV_NAME = "original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40"
_CHUNK_NAME = _PREV_NAME + "_fixedctrlinit"
_PARENT_CANDIDATES = [
    Path(r"H:\My Drive\datasets_gnn2"),
    Path(r"K:\My Drive\datasets_gnn2"),
    Path("/content/drive/MyDrive/datasets_gnn2"),
    Path(r"C:\Users\alita\OneDrive\Desktop\GNN2\datasets_gnn2"),
]
# Prefer the parent that already holds the previous dataset folder.
DATASETS_GNN2 = next(
    (p for p in _PARENT_CANDIDATES if (p / _PREV_NAME).is_dir()),
    next((p for p in _PARENT_CANDIDATES if p.is_dir()), _PARENT_CANDIDATES[0]),
)
CHUNK_ROOT = DATASETS_GNN2 / _CHUNK_NAME
CHUNK_ROOT.mkdir(parents=True, exist_ok=True)
print("PREVIOUS dataset:", DATASETS_GNN2 / _PREV_NAME)
print("THIS dataset:    ", CHUNK_ROOT)

CANDIDATES = [
    r"C:\Users\alita\OneDrive\Desktop\GNN2",
    "/content/GNN-Sandia",
    "/content/GNN2",
    os.getcwd(),
]

script_path = None
for root in CANDIDATES:
    p = os.path.join(root, "run_original_style_dataset_8500_unbalanced.py")
    if os.path.isfile(p):
        script_path = os.path.abspath(p)
        break

if script_path is None:
    raise FileNotFoundError("run_original_style_dataset_8500_unbalanced.py not found in candidates.")

os.chdir(os.path.dirname(script_path))
print("CWD:", os.getcwd())
print("SCRIPT:", script_path)
print("CHUNK_ROOT:", CHUNK_ROOT)
print("FIXED_CONTROLLER_INIT:", FIXED_CONTROLLER_INIT)

ns = {"__name__": "run_original_style_dataset_8500_unbalanced", "__file__": script_path}
exec(open(script_path, encoding="utf-8").read(), ns)

root = CHUNK_ROOT
if not root.exists():
    raise FileNotFoundError(f"CHUNK_ROOT not found: {root}")

n_chunks = math.ceil(TOTAL_SCENARIOS / SCENARIOS_PER_CHUNK)
print(f"\nTotal chunks: {n_chunks} | INCLUDE_BESS={INCLUDE_BESS} | FIXED_CONTROLLER_INIT={FIXED_CONTROLLER_INIT}")

for chunk_idx in range(n_chunks):
    s0 = chunk_idx * SCENARIOS_PER_CHUNK
    n_this = min(SCENARIOS_PER_CHUNK, TOTAL_SCENARIOS - s0)
    chunk_seed = int(BASE_SEED + 100003 * (chunk_idx + 1))
    out_dir = CHUNK_ROOT / f"run_{chunk_idx+1:03d}_scen_{s0:04d}_{s0+n_this-1:04d}_seed_{chunk_seed}"
    out_dir.mkdir(parents=True, exist_ok=True)

    ns["OUT_DIR"] = out_dir
    ns["EDGE_CSV"] = out_dir / "gnn_edges_phase_static.csv"
    ns["NODE_CSV"] = out_dir / "gnn_node_features_and_targets.csv"
    ns["SAMPLE_CSV"] = out_dir / "gnn_sample_meta.csv"
    ns["NODE_INDEX_CSV"] = out_dir / "gnn_node_index_master.csv"

    print(f"\n=== Chunk {chunk_idx+1}/{n_chunks} ===")
    print(f"Scenarios in chunk: {n_this}")
    print(f"Seed: {chunk_seed}")
    print(f"OUT_DIR: {out_dir}")

    gen_kw = dict(
        n_scenarios=int(n_this),
        k_snapshots_per_scenario_total=int(N_SAMPLES_PER_SCENARIO),
        bins_by_profile={"load": 3, "pv": 3, "net": 3},
        include_anchors=True,
        master_seed=int(chunk_seed),
        sigma_load=float(SIGMA_LOAD),
        sigma_pv=float(SIGMA_PV),
        p_load_mean_kw=float(P_LOAD_MEAN_KW),
        q_load_mean_kvar=float(Q_LOAD_MEAN_KVAR),
        p_load_scale_range=tuple(P_LOAD_SCALE_RANGE),
        q_load_scale_range=tuple(Q_LOAD_SCALE_RANGE),
        p_pv_scale_range=tuple(P_PV_SCALE_RANGE),
        vmin_safe_pu=float(VMIN_SAFE_PU),
        vmax_safe_pu=float(VMAX_SAFE_PU),
        include_source_in_safe_band=True,
        return_node_df=bool(RETURN_NODE_DF),
        node_pe_k=int(NODE_PE_K),
        node_pe_seed=int(NODE_PE_SEED),
        node_pe_zero_eig_tol=float(NODE_PE_ZERO_EIG_TOL),
        include_bess=bool(INCLUDE_BESS),
        fixed_controller_init=bool(FIXED_CONTROLLER_INIT),
    )

    if INCLUDE_BESS:
        if not BESS_CANDIDATE_NODES_150:
            raise ValueError("INCLUDE_BESS=True requires a non-empty BESS_CANDIDATE_NODES_150 list.")
        gen_kw.update(
            bess_total_mva_mean=float(BESS_TOTAL_MVA_MEAN),
            bess_total_mva_sigma=float(BESS_TOTAL_MVA_SIGMA),
            bess_num_nodes_min=int(BESS_NUM_NODES_MIN),
            bess_num_nodes_max=int(BESS_NUM_NODES_MAX),
            bess_q_frac_max=float(BESS_Q_FRAC_MAX),
            bess_candidate_nodes_override=BESS_CANDIDATE_NODES_150,
        )

    t0 = time.time()
    df_sample, df_node = ns["generate_original_style_dataset_8500_unbalanced"](**gen_kw)

    dt = time.time() - t0
    n_kept = int(df_sample["sample_id"].nunique()) if len(df_sample) else 0
    print(f"Chunk done in {dt/60:.1f} min | kept samples: {n_kept} | rows sample_meta: {len(df_sample)}")
    if len(df_sample) and "fixed_controller_init" in df_sample.columns:
        print(
            "fixed_controller_init column:",
            sorted(df_sample["fixed_controller_init"].astype(int).unique().tolist()),
        )
    print("Saved:")
    print(" -", ns["NODE_INDEX_CSV"])
    print(" -", ns["EDGE_CSV"])
    print(" -", ns["SAMPLE_CSV"])
    print(" -", ns["NODE_CSV"])

print("\nAll chunks finished.")


## 8500 — draft Y-edge sibling (from fixed-controller-init chunks)

Run **after** the chunked fixed-init generation cell finishes.

Creates a **new** folder next to the fixed-init dataset (does **not** overwrite it):

| | Path |
|--|--|
| Source | .../original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40_fixedctrlinit |
| Output | .../original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40_fixedctrlinit_yedges |

**What changes:** gnn_edges_phase_static.csv -> draft network Y (R_full/X_full = Re(Y_ij)/Im(Y_ij); edges where Y_ij != 0).

**Unchanged:** node features, PE, meta, targets (including fixed-controller-init labels).

Prefer **local** (OpenDSS) -> Drive sync -> train on Colab with CHUNK_PARENT = the *_yedges folder.


In [ ]:
# ============================================================
# 8500 Y-edge sibling from FIXEDCTRLINIT chunks (stamp_network_y_edges)
# Source:  ..._2000_40_fixedctrlinit
# Output:  ..._2000_40_fixedctrlinit_yedges
# ============================================================
from __future__ import annotations

import importlib
import sys
from pathlib import Path


def _find_repo(mod: str = "stamp_network_y_edges.py") -> Path:
    cands: list[Path] = []
    try:
        cands.append(Path(__file__).resolve().parent)
    except NameError:
        pass
    cwd = Path.cwd().resolve()
    cands.append(cwd)
    cands.extend(cwd.parents)
    cands.extend(
        [
            Path(r"C:\Users\alita\OneDrive\Desktop\GNN2"),
            Path("/content/GNN-Sandia"),
            Path("/content/GNN2"),
        ]
    )
    seen: set[Path] = set()
    for root in cands:
        root = Path(root).resolve()
        if root in seen or not root.exists():
            continue
        seen.add(root)
        if (root / mod).is_file():
            return root
    raise FileNotFoundError(f"Could not find {mod}. Open notebook from GNN2 / GNN-Sandia repo.")


_REPO = _find_repo()
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))
print("REPO:", _REPO)

import stamp_network_y_edges as stamp

stamp = importlib.reload(stamp)

SRC_NAME = "original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40_fixedctrlinit"
OUT_NAME = SRC_NAME + "_yedges"

_PARENT_CANDIDATES = [
    Path(r"H:\My Drive\datasets_gnn2"),
    Path(r"K:\My Drive\datasets_gnn2"),
    Path("/content/drive/MyDrive/datasets_gnn2"),
    _REPO / "datasets_gnn2",
]

try:
    import google.colab  # noqa: F401

    ON_COLAB = True
except ImportError:
    ON_COLAB = False

DATA = next((p for p in _PARENT_CANDIDATES if (p / SRC_NAME).is_dir()), None)
if DATA is None:
    DATA = next((p for p in _PARENT_CANDIDATES if p.is_dir()), _PARENT_CANDIDATES[0])

SRC = DATA / SRC_NAME
OUT = DATA / OUT_NAME

print("ON_COLAB:", ON_COLAB)
print("SRC:", SRC)
print("OUT:", OUT)

if not SRC.is_dir():
    raise FileNotFoundError(
        f"Source chunk parent missing:\n  {SRC}\n"
        "Finish the FIXEDCTRLINIT chunked generation cell first, then re-run this stamp."
    )

if ON_COLAB:
    print(
        "NOTE: This stamp needs OpenDSS + feeder DSS in the repo. "
        "Prefer running locally to H:/K:, then train on Colab."
    )

info = stamp.stamp_chunk_parent(
    feeder="8500",
    chunk_parent=SRC,
    out_chunk_parent=OUT,
    inplace=False,
    dry_run=False,
)
print("Done:", info)
print("Next: point training CHUNK_PARENT to:", OUT)
